In [51]:
# Core
import torch
import numpy as np

# Models
from models.hf_model import HFModel
from models.outputs import ModelOutput

# Uncertainty
from uncertainty.whitebox_uncertainty import WhiteBoxUncertainty
from uncertainty.graybox_uncertainty import GrayBoxUncertainty
from uncertainty.black_uncertainty import BlackBoxUncertainty

# Evaluation
from evaluation.aggregation import Aggregator
from evaluation.hallucination_score import HallucinationScore
from evaluation.thresholds import HallucinationThresholds
from evaluation.report import EvaluationReport

# Decision
from decision.final_score import FinalScore
from decision.hallucination_decider import HallucinationDecider


In [52]:
model = HFModel("EleutherAI/gpt-neo-125M")

  # token yoksa open model

prompt = "What is the capital city of Turkey?"

output = model.generate(
    prompt,
    max_new_tokens=10,
    num_return_sequences=3,
)

output.responses


Loading weights: 100%|██████████| 160/160 [00:00<00:00, 577.31it/s, Materializing param=transformer.wte.weight]                         
GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-125M
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0, 2, 4, 6, 8, 10}.attn.attention.bias | UNEXPECTED |  | 
transformer.h.{0...11}.attn.attention.masked_bias     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


['What is the capital city of Turkey?\n\nThe capital city of Turkey is Turkey�',
 'What is the capital city of Turkey? It is not Turkey.”\n\nWhat',
 'What is the capital city of Turkey?\n\nCity of Turkey, Turkey and a city']

In [53]:
if output.has_whitebox():
    white = WhiteBoxUncertainty(
        scores=output.logits,
        token_ids=output.token_ids,
        text_responses=output.responses
    )

    white_entropy = white.predictive_entropy()
    white_conf = white.confidence()
    white_cons = white.self_consistency()

    print("White Entropy:", white_entropy)
    print("White Confidence:", white_conf)
    print("White Consistency:", white_cons)
else:
    white_entropy = None


White Entropy: 1.3727947473526
White Confidence: 0.0027615675538547936
White Consistency: 0.3333333333333333


In [54]:
gray = GrayBoxUncertainty(
    responses=output.responses,
    log_probs=output.log_probs
)

gray_conf = gray.confidence()
gray_entropy = gray.response_entropy()

print("Gray Confidence:", gray_conf)
print("Gray Entropy:", gray_entropy)


Gray Confidence: 0.061806627883185174
Gray Entropy: 1.0986122886651097


In [55]:
black = BlackBoxUncertainty(output.responses)

black_conf = black.confidence()
black_entropy = black.response_entropy()
black_unique = black.unique_ratio()

print("Black Confidence:", black_conf)
print("Black Entropy:", black_entropy)
print("Black Unique Ratio:", black_unique)


Black Confidence: 0.3333333333333333
Black Entropy: 1.0986122886651097
Black Unique Ratio: 1.0


In [56]:
consistency = Aggregator.consistency_score(output.responses)
unique_ratio = Aggregator.unique_ratio(output.responses)

print("Aggregator Consistency:", consistency)
print("Aggregator Unique Ratio:", unique_ratio)


Aggregator Consistency: 0.3333333333333333
Aggregator Unique Ratio: 1.0


In [57]:
hs = HallucinationScore(
    white_score=white_entropy if output.has_whitebox() else None,
    gray_score=gray_conf,
    black_score=black_conf,
)

hallucination_score = hs.score()
hallucination_score


0.5912815914232525

In [58]:
risk_level = HallucinationThresholds.interpret(hallucination_score)
risk_level


'MEDIUM'

In [59]:
metrics = {
    "entropy": white_entropy if white_entropy is not None else 0,
    "self_consistency": black_conf,
    "confidence": gray_conf
}

final_score = FinalScore().compute(metrics)

decider = HallucinationDecider(
    thresholds={"hallucination": 0.6}
)

decision = decider.decide(
    {"final_score": final_score}
)

final_score, decision


(0.6948125578510104, 'hallucination')

In [60]:
report = EvaluationReport(
    hallucination_score=final_score,
    level=risk_level,
    white=white_entropy,
    gray=gray_conf,
    black=black_conf
)

report.pretty_print()
report.to_dict()


--- Evaluation Report ---
Hallucination Score : 0.6948125578510104
Risk Level : MEDIUM
White-box Entropy   : 1.3728
Gray-box Confidence : 0.0618
Black-box Consist.  : 0.3333


{'hallucination_score': 0.6948125578510104,
 'risk_level': 'MEDIUM',
 'white_uncertainty': 1.3727947473526,
 'gray_uncertainty': 0.061806627883185174,
 'black_uncertainty': 0.3333333333333333}